# Sub-basin gauge value on large basins (Kaggle)

Tests whether rain gauges add more to a **routed sub-basin model** than to a single whole-basin model, in three
large basins with public data only: Iowa River at Marengo, IA (05453100, 7,185 km²), Flint River, GA
(02347500, 4,778 km²) and La Moine River at Ripley, IL (05585000, 3,336 km²). Each basin is split into four
distance-band sub-basins with their own Daymet weather; three seeds per model.

**Setup:** Settings → *Accelerator: GPU T4 x2* and *Internet: on*, then **Save Version → Save & Run All (Commit)**.
About 2–3 hours. Download `subbasin_results.zip` from the Output tab when it finishes.

In [ ]:
import torch, shutil
from pathlib import Path
print("GPU:", torch.cuda.is_available())
!rm -rf /kaggle/temp/multigauge-hydro && git clone -q https://github.com/ogunbajoadeyinka/multigauge-hydro.git /kaggle/temp/multigauge-hydro
!pip install -q -e "/kaggle/temp/multigauge-hydro[download]"
import sys; sys.path.insert(0, "/kaggle/temp/multigauge-hydro/src")
REPO = Path("/kaggle/temp/multigauge-hydro"); ROOT = Path("/kaggle/working/sub"); ROOT.mkdir(parents=True, exist_ok=True)
# Iowa and Flint basin data are in the repository; La Moine is downloaded by the script
for site in ["05453100", "02347500"]:
    if not (ROOT / site / "data").exists():
        shutil.copytree(REPO / "validation" / site / "data", ROOT / site / "data")
print(sorted(p.name for p in ROOT.iterdir()))

In [ ]:
!cd /kaggle/temp/multigauge-hydro && python scripts/run_subbasins.py /kaggle/working/sub 05453100 02347500 05585000 --seeds 3 --epochs 10

In [ ]:
!cd /kaggle/working && rm -f subbasin_results.zip && zip -qr subbasin_results.zip sub -x "*Z?.csv" "*forcings.csv" "*rainfall.csv" "*streamflow.csv" && ls -la subbasin_results.zip
import pandas as pd; print(pd.read_csv("/kaggle/working/sub/subbasin_summary.csv").T.to_string())